# Trinity TSCP MC Task - kaggle_benchmarks v2026

**Trinity Social Cognition Probe - Multiple Choice**

Tests: Theory of mind, Pragmatic inference, Audience adaptation, Social norms

Dataset: `playra/trinity-cognitive-probes`

In [ ]:
# CELL 1: Install & Fix
!pip install protobuf==5.29.6 --quiet
!pip install -q kaggle-benchmarks kaggle

In [ ]:
# CELL 2: Imports & Config
import os
os.environ["RENDER_SUBRUNS"] = "False"

import kaggle_benchmarks as kbench
import kaggle
import pandas as pd
from dataclasses import dataclass
import glob

print("✅ Imports successful")

In [ ]:
# CELL 3: Download Dataset
!mkdir -p /kaggle/working/datasets

kaggle.api.dataset_download_files(
    'playra/trinity-cognitive-probes',
    path='/kaggle/working/datasets/',
    unzip=True
)

In [ ]:
# CELL 4: Load MC Data for TSCP
csv_files = glob.glob('/kaggle/working/datasets/**/*.csv', recursive=True)
csv_path = [f for f in csv_files if 'tscp_mc.csv' in f][0]

df = pd.read_csv(csv_path)
# Only use MC questions (skip factual)
df = df[df['question_type'] == 'mc'].copy()
eval_df = df.rename(columns={"question": "question", "choices": "choices", "answer": "expected_answer"})

print(f"📊 Loaded {len(eval_df)} TSCP MC questions")

In [ ]:
# CELL 5: Structured Output Schema
@dataclass
class MCAnswer:
    answer: str

print("✅ Schema defined")

In [ ]:
# CELL 6: Inner Task (per-item)
@kbench.task(name="TSCP MC Single", store_task=False)
def tscp_single_mc(llm, question: str, choices: str, expected_answer: str) -> bool:
    prompt = f"""{question}

{choices}

Answer with ONLY ONE letter (A, B, C, or D):"""
    resp = llm.prompt(prompt, schema=MCAnswer)
    got = resp.answer.strip().upper()[0]
    exp = str(expected_answer).strip().upper()[0]
    return got == exp

print("✅ Inner task registered")

In [ ]:
# CELL 7: Outer Benchmark Task
@kbench.task(name="Trinity TSCP MC", description="Trinity Social Cognition Probe - Multiple Choice")
def tscp_mc_benchmark(llm) -> float:
    with kbench.client.enable_cache():
        runs = tscp_single_mc.evaluate(
            llm=[llm],
            evaluation_data=eval_df,
            n_jobs=2,
            timeout=180,
            max_attempts=1,
            remove_run_files=True,
        )
    df_res = runs.as_dataframe()
    valid = df_res[df_res["result"].notna()]
    acc = float(valid["result"].mean())

    kbench.assertions.assert_true(
        True,
        expectation=f"TSCP MC accuracy: {acc:.2%} ({len(valid)}/{len(eval_df)})"
    )
    return acc

print("✅ Outer benchmark task registered")

In [ ]:
# CELL 8: Run & Choose
run = tscp_mc_benchmark.run()
print(f"\n🏆 Result: {run.result:.2%}")
%choose tscp_mc_benchmark